# 0. Node Mapping (Memory-Efficient Version)

This notebook creates LMDB databases to map raw node IDs to contiguous integer IDs.

**Optimizations:**
- Uses in-memory Python `set()` for deduplication (avoids slow LMDB lookups on HDD)
- Processes files one by one to control memory usage
- Sequential LMDB writes (no random reads)

In [1]:
# Configuration
import os

# EXTERNAL DRIVE CONFIGURATION
ROOT_DIR = "/Volumes/Backup Plus/Zaman/graph"
DATA_DIR = os.path.join(ROOT_DIR, "data")
OUTPUT_LMDB_DIR = os.path.join(ROOT_DIR, "lmdb_node_mapping")

# Dictionary mapping Node Type -> Parquet Directory
NODE_TABLES = {
    "nasabah": os.path.join(DATA_DIR, "node_nasabah"),
    "pekerja": os.path.join(DATA_DIR, "node_pekerja"),
    "pinjaman": os.path.join(DATA_DIR, "node_pinjaman"),
    "simpanan": os.path.join(DATA_DIR, "node_simpanan"),
    "transaksi": os.path.join(DATA_DIR, "node_transaksi"),
}

# Dictionary mapping Node Type -> Unique ID Column Name in Parquet
ID_COLUMN = {
    "nasabah": "cif",
    "pekerja": "pn",
    "pinjaman": "acctno",
    "simpanan": "acctno",
    "transaksi": "id_trx"
}

# Estimated Map Size (Bytes) for LMDB
MAP_SIZE = {
    "nasabah": 1024 * 1024 * 1024 * 25,
    "pekerja": 1024 * 1024 * 1024 * 1,
    "pinjaman": 1024 * 1024 * 1024 * 5,
    "simpanan": 1024 * 1024 * 1024 * 40,
    "transaksi": 1024 * 1024 * 1024 * 10
}

LMDB_COMMIT_BATCH = 500_000

os.makedirs(OUTPUT_LMDB_DIR, exist_ok=True)

In [2]:
# Imports
import lmdb
import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import json
import shutil
import time
import glob
import gc

In [3]:
def encode(x) -> bytes:
    return str(x).encode()

def format_time(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    elif seconds < 3600:
        return f"{seconds/60:.1f}m"
    else:
        return f"{seconds/3600:.1f}h"

In [4]:
def build_node_mapping_efficient():
    """Build node mappings with memory-efficient approach.
    
    Strategy:
    1. Read parquet files one by one (memory efficient)
    2. Use Python set for deduplication (fast, in-memory)
    3. Write to LMDB sequentially (no random reads)
    """
    counters = {}
    total_start = time.time()
    
    for node_type, folder in NODE_TABLES.items():
        if not os.path.exists(folder):
            print(f"Warning: Skipping {node_type} - Folder not found: {folder}")
            continue
        
        print(f"\n{'='*60}")
        print(f"Processing Node: {node_type}")
        print(f"{'='*60}")
        
        node_start = time.time()
        id_col = ID_COLUMN[node_type]
        
        # Get all parquet files
        parquet_files = glob.glob(f"{folder}/**/*.parquet", recursive=True)
        if not parquet_files:
            parquet_files = glob.glob(f"{folder}/*.parquet")
        
        if not parquet_files:
            print(f"   No parquet files found in {folder}")
            continue
        
        print(f"   Found {len(parquet_files)} parquet files")
        
        # Step 1: Collect unique IDs using Python set
        print(f"Step 1/3: Collecting unique IDs...")
        read_start = time.time()
        
        unique_ids = set()
        
        for pq_file in tqdm(parquet_files, desc="Reading files"):
            try:
                # Read only the ID column
                table = pq.read_table(pq_file, columns=[id_col])
                ids = table[id_col].to_pylist()
                
                # Add to set (converts to string)
                for id_val in ids:
                    if id_val is not None:
                        unique_ids.add(str(id_val))
                
                # Free memory
                del table, ids
                
            except Exception as e:
                print(f"\n   Error reading {pq_file}: {e}")
                continue
        
        # Force garbage collection
        gc.collect()
        
        read_time = time.time() - read_start
        print(f"   Found {len(unique_ids):,} unique IDs in {format_time(read_time)}")
        
        if len(unique_ids) == 0:
            print(f"   Warning: No IDs found for {node_type}")
            continue
        
        # Convert to list for indexing
        unique_ids_list = list(unique_ids)
        del unique_ids  # Free set memory
        gc.collect()
        
        print(f"   Sample IDs: {unique_ids_list[:3]}")
        
        # Step 2: Prepare LMDB
        print(f"Step 2/3: Preparing LMDB database...")
        lmdb_path = os.path.join(OUTPUT_LMDB_DIR, f"{node_type}.lmdb")
        
        if os.path.exists(lmdb_path):
            try:
                shutil.rmtree(lmdb_path, ignore_errors=True)
                print(f"   Removed existing database")
            except Exception:
                pass
        
        env = lmdb.open(
            lmdb_path,
            map_size=MAP_SIZE.get(node_type, 1024**3),
            subdir=True,
            lock=True,
            readonly=False,
            max_dbs=1,
        )
        
        # Step 3: Write to LMDB
        print(f"Step 3/3: Writing {len(unique_ids_list):,} entries to LMDB...")
        write_start = time.time()
        
        txn = env.begin(write=True)
        
        for i, node_id in enumerate(tqdm(unique_ids_list, desc=f"Writing {node_type}")):
            txn.put(encode(node_id), encode(i))
            
            if (i + 1) % LMDB_COMMIT_BATCH == 0:
                txn.commit()
                txn = env.begin(write=True)
        
        txn.commit()
        env.close()
        
        write_time = time.time() - write_start
        node_time = time.time() - node_start
        
        counters[node_type] = len(unique_ids_list)
        
        # Free list memory
        del unique_ids_list
        gc.collect()
        
        print(f"\n✓ Completed {node_type}:")
        print(f"   - Unique nodes: {counters[node_type]:,}")
        print(f"   - Read time: {format_time(read_time)}")
        print(f"   - Write time: {format_time(write_time)}")
        print(f"   - Total time: {format_time(node_time)}")
        print(f"   - Saved to: {lmdb_path}")
    
    total_time = time.time() - total_start
    print(f"\n{'='*60}")
    print(f"All node mappings completed in {format_time(total_time)}")
    print(f"{'='*60}")
    
    return counters

In [ ]:
if __name__ == "__main__":
    node_counts = build_node_mapping_efficient()
    
    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    
    total = 0
    for node_type, count in node_counts.items():
        print(f"{node_type:12s}: {count:>15,} nodes")
        total += count
    
    print("-"*30)
    print(f"{'TOTAL':12s}: {total:>15,} nodes")


Processing Node: nasabah
   Found 13 parquet files
Step 1/3: Collecting unique IDs...


Reading files:   0%|          | 0/13 [00:00<?, ?it/s]

   Found 12,270,075 unique IDs in 2.0m
   Sample IDs: ['HRP8991', 'SGHMT70', 'ICD2787']
Step 2/3: Preparing LMDB database...
   Removed existing database
Step 3/3: Writing 12,270,075 entries to LMDB...


Writing nasabah:   0%|          | 0/12270075 [00:00<?, ?it/s]


✓ Completed nasabah:
   - Unique nodes: 12,270,075
   - Read time: 2.0m
   - Write time: 3.4m
   - Total time: 5.5m
   - Saved to: /Volumes/Backup Plus/Zaman/graph/lmdb_node_mapping/nasabah.lmdb

Processing Node: pekerja
   Found 13 parquet files
Step 1/3: Collecting unique IDs...


...

Reading files:   0%|          | 0/13 [00:00<?, ?it/s]

   Found 15,636,712 unique IDs in 4.2m
   Sample IDs: ['474201019605504', '460801005934525', '735001000048507']
Step 2/3: Preparing LMDB database...
   Removed existing database
Step 3/3: Writing 15,636,712 entries to LMDB...


Writing simpanan:   0%|          | 0/15636712 [00:00<?, ?it/s]


✓ Completed simpanan:
   - Unique nodes: 15,636,712
   - Read time: 4.2m
   - Write time: 7.8m
   - Total time: 12.1m
   - Saved to: /Volumes/Backup Plus/Zaman/graph/lmdb_node_mapping/simpanan.lmdb

Processing Node: transaksi
   Found 13 parquet files
Step 1/3: Collecting unique IDs...


Reading files:   0%|          | 0/13 [00:00<?, ?it/s]

## Verify Mappings

In [ ]:
def verify_mapping(node_type: str, sample_size: int = 5):
    lmdb_path = os.path.join(OUTPUT_LMDB_DIR, f"{node_type}.lmdb")
    
    if not os.path.exists(lmdb_path):
        print(f"LMDB not found: {lmdb_path}")
        return
    
    env = lmdb.open(lmdb_path, readonly=True, lock=False)
    
    with env.begin() as txn:
        cursor = txn.cursor()
        
        print(f"\n{node_type} mapping samples:")
        print("-" * 50)
        
        count = 0
        for key, value in cursor:
            if count >= sample_size:
                break
            print(f"  {key.decode()} -> {value.decode()}")
            count += 1
        
        total = txn.stat()['entries']
        print(f"\nTotal entries: {total:,}")
    
    env.close()

for node_type in NODE_TABLES.keys():
    verify_mapping(node_type)